# Chapter 3: Linear Regression


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Linear regression is the oldest method in this book and, for the purposes of
learning the subject, by far the most valuable.  It is the only family of
models for which every question we shall want to ask of a machine learning
method can be answered in closed form: we can write down the optimal
parameters, their bias, their variance, their sensitivity to the data, and the
exact effect of regularisation, all with the linear algebra of
Chapter 1 and the statistics of Chapter 2.
Everything that follows in this book -- logistic regression, neural networks,
tree ensembles -- introduces complications that make one or more of these
questions intractable.  It is worth understanding thoroughly the one case
where nothing is hidden.

The chapter proceeds from the general to the particular.  We set up the linear
model and derive ordinary least squares from three independent starting points
-- as an optimisation problem, as an orthogonal projection, and as maximum
likelihood under Gaussian noise -- and then examine the statistical properties
of the resulting estimator.  We then add regularisation, first the $2$-norm
penalty that gives Ridge regression and then the $1$-norm penalty that gives
the Lasso, and show that the two behave very differently for reasons that are
entirely geometric.  A Bayesian reading unifies all three as maximum-a-posteriori
estimates under different priors.  We close with the practical questions of
scaling and the intercept, and with a complete worked analysis of the Franke
function.


## The regression problem

We are given $n$ observations of $p$ features, collected in the design matrix
$\bm{X}\in\mathbb{R}^{n\times p}$ of Eq. (1.1), together
with $n$ targets $\bm{y}\in\mathbb{R}^{n}$.  We assume the targets are
generated by

$$
\bm{y} = f(\bm{x}) + \bm{\varepsilon},
  \qquad \varepsilon_i\sim\mathcal{N}(0,\sigma^{2}),\tag{3.1}
$$

with $f$ an unknown function and $\bm{\varepsilon}$ independent noise, exactly
as in Eq. (2.44).  Our task is to construct an approximation
$\tilde{\bm{y}}$ to $f$ from the data.

**What makes a model good?.** 
It is worth being explicit, because the answer is not obvious and the obvious
answer is wrong.  A good model is *not* one that reproduces the training
data accurately; by Section *Training error, test error and generalisation* that is trivial to
achieve and tells us nothing.  A good model is one that predicts well on data
it has not seen, and by the decomposition (2.47) its
expected error on such data is the sum of a squared bias, a variance and an
irreducible noise term.  The entire content of this chapter is the management
of the first two.

Our approach throughout is *frequentist*: we treat the parameters
$\bm{\theta}$ as fixed but unknown quantities and the data as random, so that
statements about uncertainty are statements about what would happen if the
data were collected again.  Section *A Bayesian reading* shows what the same
methods look like from a Bayesian viewpoint, in which the parameters are
random and the data are fixed, and the two readings turn out to produce
identical formulae from different premises.


## The linear model and the design matrix

The model we shall study throughout is linear in the parameters,

$$
\tilde{\bm{y}} = \bm{X}\bm{\theta},
  \qquad
  \tilde{y}_i = \sum_{j=0}^{p-1}x_{ij}\theta_j = \bm{X}_{i,\ast}\bm{\theta},\tag{3.2}
$$

with $\bm{X}_{i,\ast}$ the $i$th row.  The word *linear* refers to the
parameters and not to the features, and this distinction is the source of most
of the method's usefulness.  Nothing prevents the columns of $\bm{X}$ from
being non-linear functions of some underlying variable.  If we have a single
input $x$ and choose

$$
\bm{X} =
  \begin{bmatrix}
    1 & x_0 & x_0^{2} & \cdots & x_0^{p-1}\\
    1 & x_1 & x_1^{2} & \cdots & x_1^{p-1}\\
    \vdots & \vdots & \vdots & \ddots & \vdots\\
    1 & x_{n-1} & x_{n-1}^{2} & \cdots & x_{n-1}^{p-1}
  \end{bmatrix},\tag{3.3}
$$

then Eq. (3.2) is a polynomial fit of degree $p-1$, and it
remains a *linear* regression problem.  More generally, replacing $x$ by
any set of basis functions $\phi_j(x)$ -- polynomials, splines, Fourier modes,
radial basis functions -- gives

$$
\tilde{y}_i = \sum_{j=0}^{p-1}\theta_j\,\phi_j(x_i),\tag{3.4}
$$

which is a *basis expansion*, and every method of this chapter applies
verbatim.  The matrix (3.3) is the Vandermonde matrix whose
appalling conditioning we met in Section *Vector and matrix norms*; the choice of basis
is therefore not only a modelling decision but a numerical one, and an
orthogonal polynomial basis is very much better behaved than the monomials.

The first column of ones deserves a name.  Its coefficient $\theta_0$ is the
*intercept*, the predicted value when every other feature vanishes, and
it will require separate treatment when we come to regularisation in
Section *Scaling, centring and the intercept*.

**Complexity.** 
The number of columns $p$ measures the flexibility of the model, and it is the
knob we turn when tracing out the bias-variance curve of
Section *The bias-variance tradeoff*.  For a polynomial in one variable of degree
$d$ we have $p=d+1$; for a polynomial of degree $d$ in two variables $x$ and
$y$, which is the case we shall need for the Franke function, every monomial
$x^{a}y^{b}$ with $a+b\le d$ appears and

$$
p = \frac{(d+1)(d+2)}{2}.\tag{3.5}
$$

A fifth-order fit in two variables therefore has $21$ parameters, and a
tenth-order fit has $66$: complexity grows quickly, and with it the danger
that $p$ approaches or exceeds $n$.


In [ ]:
import numpy as np

def design_matrix_2d(x, y, degree):
    """Design matrix for a two-dimensional polynomial of the given degree.

    Columns are the monomials x^a y^b with a + b <= degree, ordered by
    total degree.  The first column is the intercept.
    """
    x, y = np.ravel(x), np.ravel(y)
    columns = []
    for total in range(degree + 1):
        for b in range(total + 1):
            columns.append(x**(total - b) * y**b)
    return np.column_stack(columns)


# A degree-5 fit in two variables has (5+1)(5+2)/2 = 21 parameters
X = design_matrix_2d(np.random.rand(100), np.random.rand(100), degree=5)
print(X.shape)


## Ordinary least squares

The cost function of ordinary least squares (OLS) is the mean squared error
of Eq. (1.34),

$$
C(\bm{\theta})
   = \frac{1}{n}\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}
   = \frac{1}{n}\sum_{i=0}^{n-1}\left(y_i-\tilde{y}_i\right)^{2},\tag{3.6}
$$

and the optimisation problem is

$$
\min_{\bm{\theta}\in\mathbb{R}^{p}}
    \frac{1}{n}\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2} .\tag{3.7}
$$

We derived the solution in Section *The mean squared error and its derivative*: setting the
gradient (1.38) to zero gives the normal equations

$$
\bm{X}^{T}\bm{X}\,\bm{\theta} = \bm{X}^{T}\bm{y},
  \qquad
  \hat{\bm{\theta}}_{\mathrm{OLS}}
   = \left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T}\bm{y},\tag{3.8}
$$

with Hessian $2\bm{X}^{T}\bm{X}/n$, positive semi-definite by
Eq. (1.44), so that the problem is convex and the stationary
point is a global minimum -- unique precisely when the columns of $\bm{X}$ are
linearly independent.

**The geometric reading.** 
Equation (3.8) has an interpretation which is worth as much
as the algebra.  The vector $\bm{X}\bm{\theta}$ ranges over the column space of
$\bm{X}$ as $\bm{\theta}$ ranges over $\mathbb{R}^{p}$, so
Eq. (3.7) asks for the point of that subspace closest to
$\bm{y}$.  By Section *Orthonormal bases and projections* the answer is the orthogonal projection of
$\bm{y}$ onto the column space, and the projector is the *hat matrix*

$$
\bm{H} = \bm{X}\left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T},
  \qquad
  \tilde{\bm{y}} = \bm{H}\bm{y},\tag{3.9}
$$

which is symmetric and idempotent as required by
Eq. (1.9), with $\mathrm{Tr}(\bm{H})=p$ by
Eq. (1.10).  Equivalently, the normal
equations (3.8) may be read as
$\bm{X}^{T}(\bm{y}-\bm{X}\bm{\theta})=\bm{0}$: at the optimum the residual is
orthogonal to every column of the design matrix.  The fit has extracted from
$\bm{y}$ everything that lies in the span of the features and left the rest
untouched.

**How to compute it.** 
Equation (3.8) is a statement about $\hat{\bm{\theta}}$ and
not an instruction to a computer.  Forming $\bm{X}^{T}\bm{X}$ squares the
condition number, Eq. (1.118), and for a polynomial design
matrix of even modest degree this destroys most of the available precision.
The stable routes are the QR decomposition (1.12) or the singular
value decomposition, and through the latter the solution is the pseudoinverse
of Eq. (1.121),

$$
\hat{\bm{\theta}}_{\mathrm{OLS}} = \bm{X}^{+}\bm{y}
   = \sum_{i=0}^{r-1}\frac{\bm{u}_i^{T}\bm{y}}{\sigma_i}\,\bm{v}_i ,\tag{3.10}
$$

which additionally handles the rank-deficient case, returning the
minimum-norm solution when $p>n$ or when features are exactly collinear.
Equation (3.10) will be our main analytical tool in this chapter:
almost everything that follows is a statement about what happens to the
factors $1/\sigma_i$.


In [ ]:
import numpy as np

def ols(X, y):
    """Ordinary least squares through the SVD -- never the normal equations."""
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return Vt.T @ (U.T @ y / s)


def ols_normal_equations(X, y):
    """The textbook formula.  Shown for comparison; do not use it."""
    return np.linalg.pinv(X.T @ X) @ X.T @ y


```{admonition} Machine learning connection
:class: tip
The two functions above agree to
machine precision on a well-conditioned design matrix and can disagree in the
first significant digit on a badly conditioned one.  A simple experiment makes
the point: fit a polynomial of degree twelve to a hundred points on $[0,1]$
using both.  The condition number of the Vandermonde
matrix (3.3) is then of order $10^{8}$, that of
$\bm{X}^{T}\bm{X}$ of order $10^{16}$, and the second function returns
coefficients with no correct digits at all while the first is still accurate.
This is the practical content of Section *The singular value decomposition*, and it is why
`numpy.linalg.lstsq` exists.
```


## Weighted least squares and the $\chi^2$ function

The cost function (3.6) treats every observation as equally
reliable.  In the physical sciences this is rarely true: a measurement is
normally accompanied by an error estimate, and points known to ten per cent
should not constrain the fit as strongly as points known to one per cent.
Introducing the standard deviation $\sigma_i$ of measurement $i$, we define
the $\chi^2$ function

$$
\chi^{2}(\bm{\theta})
   = \frac{1}{n}\sum_{i=0}^{n-1}
     \frac{\left(y_i-\tilde{y}_i\right)^{2}}{\sigma_i^{2}}
   = \frac{1}{n}\left(\bm{y}-\bm{X}\bm{\theta}\right)^{T}
     \bm{\Sigma}^{-2}
     \left(\bm{y}-\bm{X}\bm{\theta}\right),\tag{3.11}
$$

where $\bm{\Sigma}$ is the diagonal matrix with entries $\sigma_i$.  Each
residual is measured in units of its own uncertainty, so that the terms of the
sum are comparable.

Minimising Eq. (3.11) is the same calculation as before with
a weight inserted.  Differentiating with the machinery of
Section *Four worked examples* gives

$$
\frac{\partial\chi^{2}}{\partial\bm{\theta}^{T}}
   = -\frac{2}{n}\bm{X}^{T}\bm{\Sigma}^{-2}
     \left(\bm{y}-\bm{X}\bm{\theta}\right) = \bm{0},\tag{3.12}
$$

and hence the *weighted* normal equations

$$
\hat{\bm{\theta}}
   = \left(\bm{X}^{T}\bm{\Sigma}^{-2}\bm{X}\right)^{-1}
     \bm{X}^{T}\bm{\Sigma}^{-2}\bm{y} .\tag{3.13}
$$

Defining $\bm{A}=\bm{\Sigma}^{-1}\bm{X}$ and $\bm{b}=\bm{\Sigma}^{-1}\bm{y}$
turns Eq. (3.13) into the ordinary
solution (3.8) for $\bm{A}$ and $\bm{b}$: weighted least
squares is unweighted least squares on rescaled data.  Setting all
$\sigma_i$ equal recovers OLS, which shows what the unweighted method
silently assumes -- that the noise is *homoscedastic*, of the same
variance everywhere.  When it is not, OLS remains unbiased but is no longer
the estimator of smallest variance, and Eq. (3.13) is.


## Measures of quality

Before comparing models we must agree on how to score them.  The mean squared
error

$$
\mathrm{MSE}(\bm{y},\tilde{\bm{y}})
   = \frac{1}{n}\sum_{i=0}^{n-1}\left(y_i-\tilde{y}_i\right)^{2}\tag{3.14}
$$

is the quantity we minimise and the natural quantity to report, but it carries
the squared units of the target and its numerical value therefore says nothing
on its own.  The *coefficient of determination*

$$
R^{2}(\bm{y},\tilde{\bm{y}})
   = 1 - \frac{\sum_{i=0}^{n-1}\left(y_i-\tilde{y}_i\right)^{2}}
              {\sum_{i=0}^{n-1}\left(y_i-\bar{y}\right)^{2}},
  \qquad
  \bar{y}=\frac{1}{n}\sum_{i=0}^{n-1}y_i,\tag{3.15}
$$

is dimensionless and compares the model against the best constant prediction:
$R^{2}=1$ is a perfect fit, $R^{2}=0$ means the model does no better than
predicting the mean, and negative values -- entirely possible on test data --
mean it does worse.  Note that the denominator is $n$ times the sample
variance of $\bm{y}$, so that $R^{2}$ is one minus the fraction of variance
left unexplained.

Two further measures appear in the sources of this book.  The *mean
absolute error*

$$
\mathrm{MAE}(\bm{y},\tilde{\bm{y}})
   = \frac{1}{n}\sum_{i=0}^{n-1}\left|y_i-\tilde{y}_i\right|\tag{3.16}
$$

is the $1$-norm counterpart of Eq. (3.14) and is far less sensitive
to outliers, for the reason discussed in Section *Vector and matrix norms*: squaring
gives a point at ten standard deviations a hundred times the weight of one at
a single standard deviation, whereas the absolute value gives it ten times.
The *relative error* $|y_i-\tilde{y}_i|/|y_i|$ is natural when the targets
span orders of magnitude, and dangerous when any of them is near zero.


In [ ]:
import numpy as np

def mse(y, y_tilde):
    return np.mean((y - y_tilde)**2)

def r2(y, y_tilde):
    return 1.0 - np.sum((y - y_tilde)**2) / np.sum((y - np.mean(y))**2)

def mae(y, y_tilde):
    return np.mean(np.abs(y - y_tilde))


A warning is in order.  All three are meaningful only on data held out from
fitting.  Computed on the training set, $R^{2}$ increases monotonically as
columns are added to $\bm{X}$ and reaches unity when $p=n$, whatever those
columns contain -- including pure noise.  Every number reported in this
chapter is a test quantity unless we say otherwise.


## Deriving least squares from a probability distribution

So far the squared-error cost (3.6) has been an assumption.  We
now show that it follows from a statement about the noise, which both explains
where it comes from and makes clear when it is the wrong choice.

Under the model (3.1), and given that the entries of the
design matrix are not stochastic, the target $y_i$ is Gaussian with mean
$\bm{X}_{i,\ast}\bm{\theta}$ and variance $\sigma^{2}$,

$$
y_i \sim \mathcal{N}\left(\bm{X}_{i,\ast}\bm{\theta},\,\sigma^{2}\right)
   = \frac{1}{\sqrt{2\pi\sigma^{2}}}
     \exp\left[-\frac{\left(y_i-\bm{X}_{i,\ast}\bm{\theta}\right)^{2}}
                     {2\sigma^{2}}\right],\tag{3.17}
$$

which is Eq. (2.39) written out in full.  Reading
Eq. (3.17) as a function of the parameters rather than of the
data gives the *likelihood* of observing $y_i$ given $\bm{\theta}$, and
since the observations are independent and identically distributed the
likelihood of the whole data set $\bm{D}$ is the product

$$
p(\bm{D}\mid\bm{\theta})
   = \prod_{i=0}^{n-1}\frac{1}{\sqrt{2\pi\sigma^{2}}}
     \exp\left[-\frac{\left(y_i-\bm{X}_{i,\ast}\bm{\theta}\right)^{2}}
                     {2\sigma^{2}}\right],\tag{3.18}
$$

where $\bm{D}$ denotes the domain of events, inputs and targets together; in
the one-dimensional case
$\bm{D}=[(x_0,y_0),(x_1,y_1),\dots,(x_{n-1},y_{n-1})]$.

*Maximum likelihood estimation* chooses the parameters making the
observed data most probable, that is maximising
Eq. (3.18).  Differentiating a product of $n$ terms is
awkward and numerically hazardous, since the product of many small numbers
underflows.  Because the logarithm is monotonically increasing, maximising the
likelihood is equivalent to maximising its logarithm, and we prefer to
minimise the negative of that:

$$
C(\bm{\theta}) = -\log\prod_{i=0}^{n-1}p(y_i,\bm{X}\mid\bm{\theta})
                 = -\sum_{i=0}^{n-1}\log p(y_i,\bm{X}\mid\bm{\theta}).\tag{3.19}
$$

Inserting Eq. (3.17) the sum evaluates to

$$
C(\bm{\theta}) = \frac{n}{2}\log\left(2\pi\sigma^{2}\right)
   + \frac{\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}}{2\sigma^{2}} .\tag{3.20}
$$

The first term does not involve $\bm{\theta}$, and the second is the
least-squares cost (3.6) up to a positive constant.
Differentiating therefore returns
$\bm{X}^{T}(\bm{y}-\bm{X}\bm{\theta})=\bm{0}$ and

$$
\hat{\bm{\theta}}_{\mathrm{OLS}}
   = \left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T}\bm{y}\tag{3.21}
$$

once more.

```{admonition} Machine learning connection
:class: tip
Equation (3.20) says
that *least squares is maximum likelihood for Gaussian noise*, and the
converse statement is the useful one: a different noise model gives a
different loss function.  Laplace-distributed noise gives the mean absolute
error (3.16); Bernoulli-distributed targets give the cross-entropy
loss of logistic regression; Poisson counts give the Poisson deviance.  When
we later write down a loss function for a neural network we are, whether or
not we say so, making an assumption about the distribution of the residuals.
Choosing the squared error for data with heavy-tailed noise is not a matter of
taste but an error, and it is why a handful of outliers can dominate a
least-squares fit.
```


## Statistical properties of the least-squares estimator

Because $\hat{\bm{\theta}}$ is a function of the random targets, it is itself a
random variable, and Section *The statistics of the least-squares estimator* established its first two
moments:

$$
\mathbb{E}[\hat{\bm{\theta}}_{\mathrm{OLS}}] = \bm{\theta},
  \qquad
  \var(\hat{\bm{\theta}}_{\mathrm{OLS}})
   = \sigma^{2}\left(\bm{X}^{T}\bm{X}\right)^{-1}.\tag{3.22}
$$

The estimator is unbiased *provided the linear model is correct*, and its
covariance matrix is the inverse Hessian scaled by the noise variance.  Since
$\sigma^{2}$ is unknown it is estimated from the residuals,

$$
\hat{\sigma}^{2}
   = \frac{\left\|\bm{y}-\bm{X}\hat{\bm{\theta}}\right\|_2^{2}}{n-p},\tag{3.23}
$$

the divisor $n-p$ accounting for the $p$ degrees of freedom consumed by the
fit, in the manner of Bessel's correction (2.28).  The
standard error of coefficient $j$ is then
$\hat{\sigma}\sqrt{[(\bm{X}^{T}\bm{X})^{-1}]_{jj}}$, from which a confidence
interval follows,

$$
\hat{\theta}_j \pm t_{\alpha/2,\,n-p}\;
    \hat{\sigma}\sqrt{\left[(\bm{X}^{T}\bm{X})^{-1}\right]_{jj}},\tag{3.24}
$$

with $t$ the appropriate quantile of Student's distribution, which for
$n-p$ large is simply the Gaussian quantile, $1.96$ for $95\%$.

**Reading the variance through the SVD.** 
Substituting Eq. (1.115) into Eq. (3.22) gives

$$
\var(\hat{\bm{\theta}}_{\mathrm{OLS}})
   = \sigma^{2}\bm{V}\tilde{\bm{\Sigma}}^{-2}\bm{V}^{T}
   = \sigma^{2}\sum_{i=0}^{p-1}\frac{\bm{v}_i\bm{v}_i^{T}}{\sigma_i^{2}},\tag{3.25}
$$

so the variance along the $i$th right singular direction is
$\sigma^{2}/\sigma_i^{2}$.  Together with the
expansion (3.10) this gives the complete picture of what
collinearity does.  A direction in feature space along which the data barely
vary has a small singular value; the coefficient along it is obtained by
dividing by that small number, and its variance is obtained by dividing by its
square.  The fit becomes an enormous positive coefficient on one feature
cancelling an enormous negative one on another, and the pair swings wildly
when the data are perturbed.  Nothing is wrong with the algebra; the data
simply do not determine that combination.

**The Gauss-Markov theorem.** 
Among all estimators that are both linear in $\bm{y}$ and unbiased, OLS has
the smallest variance.  This is the Gauss-Markov theorem, and it requires only
that the noise have zero mean, constant variance and be uncorrelated -- not
that it be Gaussian.  The theorem is often quoted as though it settled the
matter.  It does not, and the reason is
Eq. (2.27): the mean squared error counts bias and variance
equally, and Gauss-Markov restricts attention to the unbiased estimators only.
The moment we admit biased estimators, better ones exist.  The rest of this
chapter constructs them.


## Ridge regression

Ridge regression adds a penalty on the squared length of the parameter vector,

$$
\min_{\bm{\theta}\in\mathbb{R}^{p}}
    \left\{\frac{1}{n}\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}
      + \lambda\left\|\bm{\theta}\right\|_2^{2}\right\},
  \qquad \lambda\ge0,\tag{3.26}
$$

with $\|\bm{\theta}\|_2^{2}=\sum_j\theta_j^{2}$.  We derived the solution in
Eq. (1.42); dropping the factor $1/n$ so that $\lambda$ has
its conventional scaling,

$$
\boxed{\;
  \hat{\bm{\theta}}_{\mathrm{Ridge}}
   = \left(\bm{X}^{T}\bm{X}+\lambda\bm{I}\right)^{-1}\bm{X}^{T}\bm{y} \;}\tag{3.27}
$$

with $\bm{I}$ the $p\times p$ identity.  Ridge regression is therefore
ordinary least squares with a modified diagonal, and the modification cures
the central defect of Eq. (3.8): whatever the rank of
$\bm{X}$, the matrix $\bm{X}^{T}\bm{X}+\lambda\bm{I}$ has all eigenvalues at
least $\lambda$ and is invertible for every $\lambda>0$.  The estimator exists
even when $p>n$.

**The constrained form.** 
Equation (3.26) is the Lagrangian form of a constrained
problem: minimising the squared error subject to

$$
\sum_{j=0}^{p-1}\theta_j^{2}\le t\tag{3.28}
$$

for a finite $t>0$ gives the same solutions, with a one-to-one decreasing
correspondence between $t$ and $\lambda$.  The constraint region is a ball in
parameter space, and this geometric picture is what will distinguish Ridge
from the Lasso in Section *The Lasso*.

**Shrinkage, through the SVD.** 
The analytical content of Ridge regression is best seen in the singular basis.
Section *Ridge regression through the singular value decomposition* showed that Eq. (3.27) is

$$
\hat{\bm{\theta}}_{\mathrm{Ridge}}
   = \sum_{i=0}^{p-1}
     \frac{\sigma_i}{\sigma_i^{2}+\lambda}
     \left(\bm{u}_i^{T}\bm{y}\right)\bm{v}_i ,\tag{3.29}
$$

to be compared with the OLS expansion (3.10) in which the
coefficient was $1/\sigma_i$.  The fitted values follow from
Eq. (1.131),

$$
\tilde{\bm{y}}_{\mathrm{Ridge}}
   = \sum_{i=0}^{p-1}\bm{u}_i\bm{u}_i^{T}
     \frac{\sigma_i^{2}}{\sigma_i^{2}+\lambda}\,\bm{y} ,\tag{3.30}
$$

whereas for OLS the same expression holds with every factor replaced by one,
$\tilde{\bm{y}}_{\mathrm{OLS}}=\bm{U}\bm{U}^{T}\bm{y}$, which is the hat
matrix (3.9) in the singular basis.  Since $\lambda\ge0$,

$$
\frac{\sigma_i^{2}}{\sigma_i^{2}+\lambda}\le 1 ,\tag{3.31}
$$

so Ridge regression expresses $\bm{y}$ in the orthonormal basis $\bm{U}$ and
then *shrinks* each coordinate.  Because the singular values are ordered
descendingly, the shrinkage is mild for the leading directions and severe for
the trailing ones: precisely the directions whose coefficients
Eq. (3.25) showed to have the largest variance are the ones
suppressed.  The effective number of parameters is
$\mathrm{df}(\lambda)=\sum_i\sigma_i^{2}/(\sigma_i^{2}+\lambda)$ from
Eq. (1.132), falling smoothly from $p$ to zero.

**An instructive special case.** 
Suppose the design matrix is orthonormal, $\bm{X}^{T}\bm{X}=\bm{I}$.  Then
$\hat{\bm{\theta}}_{\mathrm{OLS}}=\bm{X}^{T}\bm{y}$ and

$$
\hat{\bm{\theta}}_{\mathrm{Ridge}}
   = \left(\bm{I}+\lambda\bm{I}\right)^{-1}\bm{X}^{T}\bm{y}
   = \frac{1}{1+\lambda}\,\hat{\bm{\theta}}_{\mathrm{OLS}} ,\tag{3.32}
$$

so every coefficient is scaled by the same factor $1/(1+\lambda)$ and the
estimator tends to zero as $\lambda\to\infty$.  Ridge shrinks all
coefficients *proportionally*; it never sets any of them exactly to zero.
Remember this when we reach the Lasso.

**Bias and variance.** 
Ridge regression is biased.  Taking the expectation of
Eq. (3.27) and using
$\mathbb{E}[\bm{y}]=\bm{X}\bm{\theta}$,

$$
\mathbb{E}\left[\hat{\bm{\theta}}_{\mathrm{Ridge}}\right]
   = \left(\bm{X}^{T}\bm{X}+\lambda\bm{I}\right)^{-1}
     \left(\bm{X}^{T}\bm{X}\right)\bm{\theta}
   \neq \bm{\theta}\tag{3.33}
$$

for any $\lambda>0$, the bias growing towards $-\bm{\theta}$ as
$\lambda\to\infty$.  Applying the transformation rule
$\var(\bm{A}\bm{y})=\bm{A}\var(\bm{y})\bm{A}^{T}$ to
Eq. (3.27) gives the variance,

$$
\var\left(\hat{\bm{\theta}}_{\mathrm{Ridge}}\right)
   = \sigma^{2}\left[\bm{X}^{T}\bm{X}+\lambda\bm{I}\right]^{-1}
     \bm{X}^{T}\bm{X}
     \left\{\left[\bm{X}^{T}\bm{X}+\lambda\bm{I}\right]^{-1}\right\}^{T},\tag{3.34}
$$

which vanishes as $\lambda\to\infty$.  Subtracting the two variances,

$$
\var(\hat{\bm{\theta}}_{\mathrm{OLS}})
  -\var(\hat{\bm{\theta}}_{\mathrm{Ridge}})
   = \sigma^{2}\left[\bm{X}^{T}\bm{X}+\lambda\bm{I}\right]^{-1}
     \left[2\lambda\bm{I}
       +\lambda^{2}\left(\bm{X}^{T}\bm{X}\right)^{-1}\right]
     \left\{\left[\bm{X}^{T}\bm{X}+\lambda\bm{I}\right]^{-1}\right\}^{T},\tag{3.35}
$$

and the middle bracket is non-negative definite for $\lambda>0$, hence so is
the whole product.  The variance of the Ridge estimator is therefore
*always* smaller than that of OLS.

This is the bias-variance trade-off of Section *The bias-variance tradeoff* in
closed form.  Increasing $\lambda$ increases the squared bias and decreases the
variance, monotonically in both cases; by Eq. (2.27) the
total error is their sum, and since the variance falls immediately while the
bias starts at zero with zero derivative, there always exists a
$\lambda>0$ giving a smaller mean squared error than OLS.  Gauss-Markov is not
contradicted -- the Ridge estimator is not unbiased, so it was never in the
competition.


## The Lasso

Replacing the $2$-norm penalty of Eq. (3.26) by a $1$-norm
gives

$$
\min_{\bm{\theta}\in\mathbb{R}^{p}}
    \left\{\frac{1}{n}\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}
      + \lambda\left\|\bm{\theta}\right\|_1\right\},
  \qquad
  \left\|\bm{\theta}\right\|_1 = \sum_{j=0}^{p-1}\left|\theta_j\right| ,\tag{3.36}
$$

which is the Lasso -- least absolute shrinkage and selection operator.  The
change looks slight and its consequences are not: the Lasso sets coefficients
exactly to zero, and thereby performs variable selection as part of the
fitting, which Ridge regression by Eq. (3.32) never
does.

**No closed form.** 
Differentiating Eq. (3.36) requires the derivative of the
absolute value,

$$
\frac{d\left|\theta\right|}{d\theta} = \mathrm{sgn}(\theta) =
  \begin{cases}
    +1 & \theta>0,\\
    -1 & \theta<0,
  \end{cases}\tag{3.37}
$$

which is undefined at the origin.  Ignoring that difficulty for a moment and
dropping the $1/n$, the stationarity condition reads

$$
-2\bm{X}^{T}\left(\bm{y}-\bm{X}\bm{\theta}\right)
    + \lambda\,\mathrm{sgn}(\bm{\theta}) = \bm{0},
  \qquad\text{that is}\qquad
  2\bm{X}^{T}\bm{X}\bm{\theta} + \lambda\,\mathrm{sgn}(\bm{\theta})
   = 2\bm{X}^{T}\bm{y} .\tag{3.38}
$$

Because $\mathrm{sgn}(\bm{\theta})$ depends on the unknown in a
non-differentiable way, Eq. (3.38) cannot be solved by
a matrix inversion, and no analogue of Eqs. (3.8) or
(3.27) exists.  The problem is nonetheless *convex* --
a sum of a convex quadratic and a convex norm -- so it has a global minimum
and can be solved reliably; it is the closed form that is lost, not the
solution.

**The orthonormal case and soft thresholding.** 
The behaviour becomes transparent in the simplest possible design.  Take
$\bm{X}=\bm{I}$ with $n=p$, so that $\tilde{\bm{y}}=\bm{\theta}$ and the
problem decouples completely into $p$ scalar problems.  For OLS,

$$
C(\bm{\theta}) = \sum_{i=0}^{p-1}\left(y_i-\theta_i\right)^{2},
  \qquad
  \hat{\theta}_i^{\mathrm{OLS}} = y_i .\tag{3.39}
$$

For Ridge,

$$
C(\bm{\theta}) = \sum_{i}\left(y_i-\theta_i\right)^{2}
                 + \lambda\sum_{i}\theta_i^{2},
  \qquad
  \hat{\theta}_i^{\mathrm{Ridge}} = \frac{y_i}{1+\lambda},\tag{3.40}
$$

in agreement with Eq. (3.32).  For the Lasso,

$$
C(\bm{\theta}) = \sum_{i}\left(y_i-\theta_i\right)^{2}
                 + \lambda\sum_{i}\left|\theta_i\right| ,\tag{3.41}
$$

and treating each $i$ separately, the derivative for $\theta_i\neq0$ is
$-2(y_i-\theta_i)+\lambda\,\mathrm{sgn}(\theta_i)=0$.  If $\theta_i>0$ this
gives $\theta_i=y_i-\lambda/2$, which is consistent with $\theta_i>0$ only
when $y_i>\lambda/2$; if $\theta_i<0$ it gives $\theta_i=y_i+\lambda/2$,
consistent only when $y_i<-\lambda/2$.  For $|y_i|\le\lambda/2$ neither branch
is consistent and the minimum lies at the kink $\theta_i=0$.  Collecting,

$$
\boxed{\;
  \hat{\theta}_i^{\mathrm{Lasso}} =
  \begin{cases}
    y_i - \lambda/2 & y_i > \lambda/2,\\[2pt]
    y_i + \lambda/2 & y_i < -\lambda/2,\\[2pt]
    0               & \left|y_i\right| \le \lambda/2 .
  \end{cases} \;}\tag{3.42}
$$

This is the *soft thresholding* operator, written compactly as

$$
S_{\lambda/2}(y) = \mathrm{sgn}(y)\,\max\left(|y|-\lambda/2,\;0\right).\tag{3.43}
$$

Compare the three results.  OLS leaves the coefficient alone.  Ridge
multiplies it by $1/(1+\lambda)$, shrinking it towards zero but reaching zero
only in the limit $\lambda\to\infty$.  The Lasso subtracts a constant
$\lambda/2$ from its magnitude and *truncates at zero*, so every
coefficient smaller than the threshold is eliminated outright at finite
$\lambda$.  The difference between shrinkage and selection is contained in
this one comparison.

**Why the geometry does it.** 
The constrained forms explain the same fact without any calculus.  Ridge
minimises the squared error subject to $\|\bm{\theta}\|_2^{2}\le t$, a ball;
the Lasso subject to $\|\bm{\theta}\|_1\le t$, a cross-polytope -- a diamond
in two dimensions.  The solution lies where the elliptical contours of the
squared error first touch the constraint region.  A ball has no distinguished
points, so the contact occurs at a generic location with all coordinates
non-zero.  A diamond has corners, and the corners lie *on the axes*,
where some coordinates vanish.  Contours are far more likely to meet a
protruding corner than a flat face, and every corner is a solution with a zero
coefficient.  In $p$ dimensions the $1$-norm ball has corners, edges and faces
of every dimension, each corresponding to a different set of variables being
eliminated.

**Coordinate descent.** 
Equation (3.42) is exact only for an orthonormal design,
but it suggests the algorithm that solves the general problem.  Suppose we fix
all coefficients but one and minimise over $\theta_j$ alone.  Writing the
partial residual with the $j$th contribution removed,

$$
\bm{r}^{(j)} = \bm{y} - \sum_{k\neq j}\bm{x}_k\theta_k ,\tag{3.44}
$$

where $\bm{x}_k$ is column $k$, the one-dimensional problem is exactly of the
form solved above, and its answer is

$$
\theta_j \leftarrow
    \frac{S_{\lambda/2}\left(\bm{x}_j^{T}\bm{r}^{(j)}\right)}
         {\bm{x}_j^{T}\bm{x}_j} ,\tag{3.45}
$$

which for standardised columns has $\bm{x}_j^{T}\bm{x}_j=n$.  Cycling through
the coordinates until the parameters stop changing is *coordinate
descent*, and because the cost is convex and the non-differentiable part is
separable -- it is a sum of terms each involving one coordinate -- the
procedure is guaranteed to converge to the global minimum.  This is what
`sklearn.linear_model.Lasso` runs.  The reader will recognise the
pattern from the Gauss-Seidel iteration of Eq. (1.92):
update one coordinate at a time, using the most recent values of all the
others.


In [ ]:
import numpy as np

def soft_threshold(z, gamma):
    """Soft thresholding operator S_gamma(z), Eq. (3.softthresholdop)."""
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0.0)


def lasso_coordinate_descent(X, y, lmbda, n_iter=1000, tol=1e-8):
    """Lasso by cyclic coordinate descent.

    Minimises ||y - X theta||^2 / n + lmbda * ||theta||_1.
    The columns of X are assumed centred and standardised, and no
    intercept is penalised -- see Section on scaling and the intercept.
    """
    n, p = X.shape
    theta = np.zeros(p)
    col_norms = np.sum(X**2, axis=0)
    r = y - X @ theta                              # full residual

    for _ in range(n_iter):
        theta_old = theta.copy()
        for j in range(p):
            # partial residual: add back the current contribution of column j
            r += X[:, j] * theta[j]
            rho = X[:, j] @ r
            theta[j] = soft_threshold(rho, lmbda * n / 2.0) / col_norms[j]
            r -= X[:, j] * theta[j]                # remove the updated one
        if np.max(np.abs(theta - theta_old)) < tol:
            break

    return theta


```{admonition} Machine learning connection
:class: tip
The selection property makes the Lasso
the natural first tool when $p$ is large and most features are believed
irrelevant -- gene expression studies, text features, high-order polynomial
bases.  It has real limitations.  When several features are strongly
correlated the Lasso tends to pick one arbitrarily and discard the rest, which
is unstable under resampling and misleading if the discarded features are
scientifically meaningful; when $p>n$ it can select at most $n$ variables.
The *elastic net*, which penalises
$\alpha\|\bm{\theta}\|_1+(1-\alpha)\|\bm{\theta}\|_2^{2}$, was designed to
repair both defects by combining selection with the grouping behaviour of
Ridge.  Note finally that the sparsity is a property of the $1$-norm, not of
regression: the same penalty produces sparse solutions in compressed sensing,
in dictionary learning and in the pruning of neural networks.
```


## Comparing the three estimators

It is worth seeing the three methods act on the same small problem.  Take the
targets and design matrix

$$
\bm{y}=\begin{bmatrix}4\\2\\3\end{bmatrix},
  \qquad
  \bm{X}=\begin{bmatrix}2&0\\0&1\\0&0\end{bmatrix},\tag{3.46}
$$

so that there are three observations, two features and two parameters.  The
third row contributes nothing to the fit but does contribute to the error,
which is what makes the example non-trivial.  Since
$\bm{X}\bm{\theta}=(2\theta_0,\theta_1,0)^{T}$, the unpenalised cost is

$$
C(\bm{\theta}) = \left(4-2\theta_0\right)^{2}
                 + \left(2-\theta_1\right)^{2} + 3^{2},\tag{3.47}
$$

minimised at

$$
\hat{\bm{\theta}}_{\mathrm{OLS}}
   = \left(\bm{X}^{T}\bm{X}\right)^{-1}\bm{X}^{T}\bm{y}
   = \begin{bmatrix}2\\2\end{bmatrix}.\tag{3.48}
$$

Adding the Ridge penalty gives
$C(\bm{\theta})=(4-2\theta_0)^{2}+(2-\theta_1)^{2}
+\lambda(\theta_0^{2}+\theta_1^{2})$ up to the constant, and differentiating
with respect to each parameter separately,

$$
\hat{\bm{\theta}}_{\mathrm{Ridge}}
   = \begin{bmatrix}
       \dfrac{8}{4+\lambda}\\[8pt]
       \dfrac{2}{1+\lambda}
     \end{bmatrix},\tag{3.49}
$$

which reproduces Eq. (3.48) at $\lambda=0$ and decays to zero as
$\lambda$ grows.  Notice that the two coefficients are shrunk by
*different* factors, $4/(4+\lambda)$ and $1/(1+\lambda)$: the penalty is
applied uniformly in parameter space, but the curvature of the cost differs
between directions, so the coefficient belonging to the better-determined
feature resists shrinkage more strongly.  This is
Eq. (3.31) with $\sigma_0^2=4$ and $\sigma_1^2=1$.  Neither
coefficient ever becomes exactly zero.

The Lasso behaves differently.  Because the problem is diagonal, each
coefficient can be treated separately as in Eq. (3.42):
differentiating $(2-\theta_1)^{2}+\lambda|\theta_1|$ gives
$\theta_1=\max(2-\lambda/2,\,0)$, while $(4-2\theta_0)^{2}+\lambda|\theta_0|$
gives $\theta_0=\max\left((16-\lambda)/8,\,0\right)$.  The first coefficient is
eliminated at $\lambda=4$ and the second at $\lambda=16$, both at finite
$\lambda$ and both exactly.  Plotting all
three coefficient paths against $\lambda$ shows the distinction at a glance:
the Ridge curves approach the axis asymptotically, the Lasso curves reach it
and stop.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso

X = np.array([[2.0, 0.0], [0.0, 1.0], [0.0, 0.0]])
y = np.array([4.0, 2.0, 3.0])

n = X.shape[0]
lambdas = np.logspace(-3, 2, 200)

# scikit-learn's Ridge minimises ||y - X t||^2 + alpha ||t||^2, so alpha = lambda,
# but its Lasso minimises ||y - X t||^2 / (2n) + alpha ||t||_1, so alpha = lambda / (2n).
ridge_path = np.array([Ridge(alpha=l, fit_intercept=False).fit(X, y).coef_
                       for l in lambdas])
lasso_path = np.array([Lasso(alpha=l / (2 * n), fit_intercept=False,
                             max_iter=100000).fit(X, y).coef_
                       for l in lambdas])

# Analytical results: Eq. (3.toyridge) for Ridge, Eq. (3.softthreshold) for Lasso
ridge_exact = np.column_stack([8.0 / (4.0 + lambdas), 2.0 / (1.0 + lambdas)])
lasso_exact = np.column_stack([np.maximum((16.0 - lambdas) / 8.0, 0.0),
                               np.maximum(2.0 - lambdas / 2.0, 0.0)])
assert np.allclose(ridge_path, ridge_exact) and np.allclose(lasso_path, lasso_exact)

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for k in range(2):
    ax[0].semilogx(lambdas, ridge_path[:, k], label=rf"$\theta_{k}$")
    ax[1].semilogx(lambdas, lasso_path[:, k], label=rf"$\theta_{k}$")
ax[0].set_title("Ridge"); ax[1].set_title("Lasso")
for a in ax:
    a.set_xlabel(r"$\lambda$"); a.legend()
plt.show()


The two rescalings in that code deserve attention, because they are exactly
the trap warned against in Section *Scaling, centring and the intercept*.
`scikit-learn` minimises
$\|\bm{y}-\bm{X}\bm{\theta}\|_2^{2}+\alpha\|\bm{\theta}\|_2^{2}$ for Ridge but
$\|\bm{y}-\bm{X}\bm{\theta}\|_2^{2}/(2n)+\alpha\|\bm{\theta}\|_1$ for the
Lasso -- the same library uses *different* conventions for the two
methods, so $\alpha=\lambda$ in one case and $\alpha=\lambda/(2n)$ in the
other.  With the conversions in place the computed paths agree with the
analytical results (3.49) and (3.42) to
machine precision, which is what the assertion checks.  Such factors differ
between every implementation and every textbook, and reconciling one's own
code with a library means checking them explicitly rather than assuming.


## A Bayesian reading

The three estimators can be derived a second time, from a viewpoint in which
the parameters rather than the data are random.  The unification is elegant,
and it explains where the penalties come from rather than merely postulating
them.

**Bayes' theorem.** 
From the product rule for joint probabilities,
$p(X,Y)=p(X\mid Y)p(Y)=p(Y\mid X)p(X)$, we obtain immediately

$$
p(\bm{\theta}\mid\bm{D})
   = \frac{p(\bm{D}\mid\bm{\theta})\,p(\bm{\theta})}{p(\bm{D})}
   \;\propto\; p(\bm{D}\mid\bm{\theta})\,p(\bm{\theta}),\tag{3.50}
$$

dropping the normalisation $p(\bm{D})$, which does not depend on
$\bm{\theta}$.  The left-hand side is the *posterior*, the probability of
the parameters given the data; on the right are the *likelihood*
$p(\bm{D}\mid\bm{\theta})$, which we already modelled in
Eq. (3.18), and the *prior* $p(\bm{\theta})$, which
encodes what we believed about the parameters before seeing any data.
Choosing the $\bm{\theta}$ that maximises the posterior is
*maximum-a-posteriori* (MAP) estimation.

**A Gaussian prior gives Ridge.** 
Suppose we believe, before seeing the data, that the parameters are small:
independent, zero-mean Gaussians of variance $\tau^{2}$,

$$
p(\bm{\theta}) = \prod_{j=0}^{p-1}
    \exp\left(-\frac{\theta_j^{2}}{2\tau^{2}}\right).\tag{3.51}
$$

The posterior is then the product of Eq. (3.18) and
Eq. (3.51).  Taking the negative logarithm and discarding
terms independent of $\bm{\theta}$,

$$
C(\bm{\theta})
   = \frac{\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}}{2\sigma^{2}}
   + \frac{1}{2\tau^{2}}\left\|\bm{\theta}\right\|_2^{2},\tag{3.52}
$$

which, on identifying $\lambda=\sigma^{2}/\tau^{2}$ after multiplying through
by $2\sigma^2$, is exactly the Ridge cost
function (3.26).

**A Laplace prior gives the Lasso.** 
Replace the Gaussian prior by a Laplace (double exponential) distribution with
zero mean,

$$
p(\bm{\theta}) = \prod_{j=0}^{p-1}
    \exp\left(-\frac{\left|\theta_j\right|}{\tau}\right).\tag{3.53}
$$

The same steps give

$$
C(\bm{\theta})
   = \frac{\left\|\bm{y}-\bm{X}\bm{\theta}\right\|_2^{2}}{2\sigma^{2}}
   + \frac{1}{\tau}\left\|\bm{\theta}\right\|_1,\tag{3.54}
$$

the Lasso cost function (3.36).  A flat prior, expressing
no preference at all, leaves only the likelihood and returns ordinary least
squares.

**What the unification tells us.** 
The strength of the prior and the strength of the penalty are the same
quantity.  A large $\lambda$ corresponds to a small prior variance -- a firm
belief that the coefficients are near zero -- and a small $\lambda$ to a
diffuse prior that lets the data speak.  The difference between Ridge and the
Lasso is now visibly a difference between two prior *shapes*: the Laplace
density has a sharp peak at the origin and heavier tails than the Gaussian, so
it simultaneously expresses a stronger belief that coefficients are exactly
zero and a greater tolerance for the few that are large.  That is precisely
the behaviour of Eq. (3.42).

It should be said that MAP estimation is not the whole of Bayesian inference.
It returns a single point, the mode of the posterior, and discards the
distribution around it -- which is the part a Bayesian would consider the
answer.  A full treatment would report the posterior itself, giving credible
intervals directly rather than through the sampling argument of
Eq. (3.24), and would integrate over $\bm{\theta}$ when making
predictions instead of fixing it at the mode.  We return to this when we
discuss Bayesian neural networks and the Gaussian processes for which the
Cholesky factorisation of Section *LU and Cholesky decompositions* was introduced.


## Scaling, centring and the intercept

We now address a set of practical questions which cause more confusion than
any of the mathematics above, and which are the usual reason a hand-written
implementation disagrees with a library.

**Why scaling matters for penalised regression.** 
The penalties $\|\bm{\theta}\|_2^{2}$ and $\|\bm{\theta}\|_1$ treat all
coefficients alike, but a coefficient's magnitude depends on the units of its
feature.  Suppose one predictor is a person's height.  Measured in
millimetres the fitted coefficient is a thousand times smaller than the same
coefficient measured in metres, and it is therefore penalised a thousand times
less.  The consequence is stark: for Ridge and the Lasso, a change of units is
not a harmless relabelling but a change of model.  Unless the features are
already on comparable scales, they must be standardised -- each column
centred and divided by its standard deviation, as in
Section *Arrays in practice: numpy, BLAS and LAPACK* -- before a penalty is applied.  Ordinary least
squares is exempt, since Eq. (3.8) is equivariant under
rescaling of the columns; the fitted values do not change, only the
coefficients, in a compensating way.

**The intercept should not be penalised.** 
The intercept $\theta_0$ is the expected target when all predictors are zero.
Penalising it shrinks the fitted values towards zero rather than towards the
mean of the data, which is almost never what is wanted and which makes the
result depend on where the origin of the target happens to be.  If we shift
every $y_i$ by a constant, we would like the fit simply to shift with it, and
that fails if $\theta_0$ is penalised.

The standard remedy is to centre both $\bm{X}$ and $\bm{y}$.  If every column
of $\bm{X}$ and the target $\bm{y}$ have had their means subtracted, the
intercept of the centred problem is exactly zero, so it can be dropped from
the design matrix altogether and the penalty then applies only to the genuine
slopes.  The intercept is recovered afterwards from

$$
\hat{\theta}_0 = \bar{y} - \sum_{j=1}^{p-1}\bar{x}_j\hat{\theta}_j ,\tag{3.55}
$$

with $\bar{y}$ and $\bar{x}_j$ the training means.  This is what
`scikit-learn` does internally when the intercept is fitted, and it is
why its default solutions are derived under the assumption that both
$\bm{y}$ and $\bm{X}$ are zero centred.


In [ ]:
import numpy as np

def fit_with_intercept(X, y, fit_centred):
    """Fit a penalised model without penalising the intercept.

    fit_centred(Xc, yc) must return coefficients for centred data with no
    intercept column.  Returns (theta_0, theta) plus the training
    statistics needed to transform future data identically.
    """
    x_mean = np.mean(X, axis=0)
    y_mean = np.mean(y)
    x_std = np.std(X, axis=0)
    x_std[x_std == 0.0] = 1.0                  # leave constant columns alone

    Xc = (X - x_mean) / x_std
    theta = fit_centred(Xc, y - y_mean)

    # Undo the scaling so the coefficients apply to the raw features
    theta = theta / x_std
    theta_0 = y_mean - x_mean @ theta
    return theta_0, theta, (x_mean, x_std, y_mean)


def predict(X_new, theta_0, theta):
    return theta_0 + X_new @ theta


**The same transformation must be applied to new data.** 
The means and standard deviations are computed on the *training* data
and then applied unchanged to validation, test and future data.  Recomputing
them on the test set, or computing them once on the full data set before
splitting, leaks information and produces an optimistic error estimate, as
warned in Sections *Training error, test error and generalisation* and *Cross-validation*.
Inside a cross-validation loop this means the scaler must be refitted on every
training fold, which is what a `Pipeline` does automatically.

```{admonition} Machine learning connection
:class: tip
A checklist for reconciling one's own
code with a library, in decreasing order of how often each is the culprit:
whether the intercept is included in the design matrix or handled separately;
whether the intercept is penalised; whether the data have been centred and
scaled, and with which convention for the standard deviation ($n$ or $n-1$);
whether the cost function carries a factor $1/n$, $1/(2n)$ or nothing, which
rescales $\lambda$ correspondingly; and whether the library's regularisation
parameter is our $\lambda$ or some multiple of it.  Every one of these changes
the numbers while leaving the method unchanged, and discovering which is
responsible is a routine part of the work.
```


## A complete example: the Franke function

We now assemble everything into a single study.  The Franke function is a
weighted sum of four exponentials on the unit square,

$$
\begin{align}
f(x,y) &= \frac{3}{4}\exp\left(-\frac{(9x-2)^{2}}{4}
                                -\frac{(9y-2)^{2}}{4}\right)
          + \frac{3}{4}\exp\left(-\frac{(9x+1)^{2}}{49}
                                -\frac{9y+1}{10}\right) \nonumber\\
         &\quad + \frac{1}{2}\exp\left(-\frac{(9x-7)^{2}}{4}
                                -\frac{(9y-3)^{2}}{4}\right)
          - \frac{1}{5}\exp\left(-(9x-4)^{2}-(9y-7)^{2}\right),
\end{align}
$$

for $x,y\in[0,1]$.  It is a standard test surface for interpolation and
regression: smooth, but with two peaks, a saddle and a depression, so that it
cannot be captured by a low-order polynomial and is well captured by a
moderate one.  Adding Gaussian noise gives us a problem with a known truth, a
known noise level and a tunable complexity -- everything needed to see the
bias-variance trade-off of Section *The bias-variance tradeoff* in action.


In [ ]:
import numpy as np

def franke_function(x, y):
    """The Franke function, Eq. (3.franke)."""
    t1 = 0.75 * np.exp(-(0.25 * (9 * x - 2)**2) - 0.25 * ((9 * y - 2)**2))
    t2 = 0.75 * np.exp(-((9 * x + 1)**2) / 49.0 - 0.1 * (9 * y + 1))
    t3 = 0.50 * np.exp(-(9 * x - 7)**2 / 4.0 - 0.25 * ((9 * y - 3)**2))
    t4 = -0.20 * np.exp(-(9 * x - 4)**2 - (9 * y - 7)**2)
    return t1 + t2 + t3 + t4


def make_franke_data(n=400, noise=0.1, rng=None):
    """Sample the Franke function on random points with Gaussian noise."""
    rng = np.random.default_rng() if rng is None else rng
    x, y = rng.random(n), rng.random(n)
    z = franke_function(x, y) + noise * rng.normal(size=n)
    return x, y, z


**The study.** 
The analysis proceeds in the order of this book.  We build the design matrix
of Eq. (3.5) for polynomial degrees from one to some
maximum; we split the data into training and test sets; we standardise using
training statistics only; we fit by OLS, Ridge and Lasso; and we report test
errors, choosing $\lambda$ by the $k$-fold cross-validation of
Section *Cross-validation*.


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score

rng = np.random.default_rng(2024)
# A small, noisy sample: with n = 100 the number of parameters (3.numfeatures)
# overtakes the 80 training points around degree eleven, which is where the
# difference between the three methods becomes visible.
x, y, z = make_franke_data(n=100, noise=0.2, rng=rng)

max_degree = 14
kfold = KFold(n_splits=5, shuffle=True, random_state=2024)
lambdas = np.logspace(-5, 1, 25)

results = {"OLS": [], "Ridge": [], "Lasso": []}
for degree in range(1, max_degree + 1):
    X = design_matrix_2d(x, y, degree)[:, 1:]      # drop the intercept column
    X_train, X_test, z_train, z_test = train_test_split(X, z, test_size=0.2,
                                                        random_state=2024)

    # OLS: the scaler is fitted on the training fold only, inside the pipeline
    ols_pipe = make_pipeline(StandardScaler(), LinearRegression())
    ols_pipe.fit(X_train, z_train)
    results["OLS"].append(mse(z_test, ols_pipe.predict(X_test)))

    # Ridge and Lasso: choose lambda by cross-validation on the training set
    for name, Model in (("Ridge", Ridge), ("Lasso", Lasso)):
        cv_error = [
            -cross_val_score(make_pipeline(StandardScaler(),
                                           Model(alpha=lmb, max_iter=5000)),
                             X_train, z_train, cv=kfold,
                             scoring="neg_mean_squared_error").mean()
            for lmb in lambdas
        ]
        best = make_pipeline(StandardScaler(),
                             Model(alpha=lambdas[int(np.argmin(cv_error))],
                                   max_iter=5000))
        best.fit(X_train, z_train)
        results[name].append(mse(z_test, best.predict(X_test)))
        if name == "Lasso":
            n_kept = int(np.sum(best[-1].coef_ != 0))

for name, errors in results.items():
    print(f"{name:>6}: best degree {int(np.argmin(errors)) + 1:2d}, "
          f"test MSE {min(errors):.4f}, "
          f"MSE at degree {max_degree} {errors[-1]:.4f}")


**What one observes.** 
The output of the run above is


```
   OLS: best degree  4, test MSE 0.0570, MSE at degree 14 2428714.9322
 Ridge: best degree  2, test MSE 0.0599, MSE at degree 14       0.0667
 Lasso: best degree  3, test MSE 0.0607, MSE at degree 14       0.0644
```


and three features of it are worth dwelling on, because they are the lessons
of the preceding chapters made visible.

First, all three methods achieve very nearly the same best test error, about
$0.057$ to $0.061$, against a noise variance of $\sigma^{2}=0.04$.  By
Eq. (2.47) that floor cannot be crossed, and the three
methods are all within about fifty per cent of it.  *Regularisation does
not make a good model better.*

Second, the behaviour away from the optimum is entirely different.  The OLS
error is well behaved up to degree eight, rises to $14.5$ at degree ten and
then to over two million at degree fourteen -- seven orders of magnitude worse
than its own best value.  The reason is Eq. (3.5): at
degree fourteen there are $119$ parameters and only $80$ training points, so
$\bm{X}^{T}\bm{X}$ is singular, the smallest singular values are pure noise,
and Eq. (3.10) divides by them.  The penalised errors, by
contrast, rise from $0.060$ to $0.067$ and from $0.061$ to $0.064$: they
barely move.  Cross-validation increases $\lambda$ as the degree grows, and
the shrinkage factors (3.31) suppress precisely the
directions in which Eq. (3.25) shows the variance to be
unbounded.  What regularisation buys is not a better optimum but insensitivity
to choosing the complexity badly -- which matters, because in a real problem
the right complexity is not known.

Third, the Lasso fits are genuinely sparse.  At degree ten it keeps $6$ of
$65$ coefficients, and at degree fourteen $7$ of $119$; Ridge at comparable
test error keeps every one of them, each small.  The Lasso has performed model
selection, discovering from the data that a handful of low-order monomials
suffices -- consistent with its optimum at degree three.

Two further experiments make the point sharper.  Increasing $n$ from $100$ to
$400$ moves the OLS optimum from degree four to degree eleven and reduces the
degradation at degree fourteen from a factor of $4\times10^{7}$ to a factor of
$3$, because $p$ no longer approaches $n$.  Reducing the noise to $\sigma=0$
while keeping $n=100$ moves the optimum from degree four to degree eight, since
with no noise there is less to overfit.  Note that even then the error
eventually grows, by a factor of about $180$ at degree fourteen: once
$p$ exceeds the number of training points the design matrix is rank deficient
and Eq. (3.10) is dividing by singular values that are pure
rounding error.  That failure is one of conditioning rather than of
statistics, and it is the only one of the three that no amount of clean data
will cure.  The best model is a property of the data set -- its size and its
noise level -- and not of the function being approximated.

```{admonition} Machine learning connection
:class: tip
The whole analysis rests on one number
being honest: the test error.  It is worth listing the ways of corrupting it,
all of which appear in student work and published papers alike.  Standardising
before the split leaks the test distribution into training.  Choosing the
polynomial degree by looking at the test error, then reporting that same error,
turns the test set into a validation set -- the reported figure is then
optimistic, and the remedy is the three-way split of
Section *Training error, test error and generalisation*.  Selecting features on the full data set
before cross-validating leaks in the same way, and dramatically so when $p$ is
large.  And splitting spatially or temporally correlated data at random makes
the test points near-duplicates of training points, which by
Section *Correlated data and the autocorrelation function* measures interpolation rather than prediction.  For
data sampled on a grid, as the Franke function often is, this last is a real
concern.
```


## Summary and the programs

Linear regression has given us, in closed form, every quantity the rest of
this book will only be able to estimate.

The estimator itself arrived three times over.  As an optimisation problem it
is the minimiser of the squared error, Eq. (3.8); as
geometry it is the orthogonal projection of the targets onto the column space
of the design matrix, Eq. (3.9); and as statistics it is the
maximum-likelihood estimate under Gaussian noise,
Eq. (3.21).  The three derivations are worth holding together,
because each generalises differently: the first survives into every method in
this book, the second is what makes the linear case special, and the third is
what tells us that a different noise model demands a different loss.

The three penalties then formed a sequence.  Ordinary least squares is
unbiased and, by Gauss-Markov, of minimum variance among unbiased linear
estimators -- and that guarantee is worth less than it sounds, because
Eq. (2.27) counts variance alongside bias.  Ridge
regression buys a large reduction in variance for a small bias, shrinking each
singular direction by $\sigma_i^{2}/(\sigma_i^{2}+\lambda)$ and hence
suppressing hardest exactly those directions in which
Eq. (3.25) shows the data to be least informative.  The Lasso
replaces proportional shrinkage by soft
thresholding (3.42), and thereby sets coefficients exactly
to zero -- a difference which is visible in the algebra, in the geometry of
the $1$-norm ball, and in the shape of the corresponding Laplace prior.

The Bayesian reading of Section *A Bayesian reading* unified the three: a flat
prior gives OLS, a Gaussian prior gives Ridge, a Laplace prior gives the
Lasso.  Regularisation is the expression of a prior belief, and the
regularisation parameter is its strength.

Finally, none of this survives careless practice.  Penalties are meaningless
unless the features are on comparable scales; the intercept must be excluded
from the penalty; every transformation must be fitted on training data alone;
and the number one reports must come from data the model has never seen.

The complete programs are collected in `doc/BookML/BookPrograms`:

- `linear_regression.py` -- OLS by the SVD, by QR and by the
   normal equations, with the conditioning comparison of
   Section *Ordinary least squares*; weighted least squares; and the quality
   measures of Section *Measures of quality*.
- `ridge_lasso.py` -- Ridge in closed form and through the SVD,
   the Lasso by coordinate descent with soft thresholding, and the
   coefficient paths of Section *Comparing the three estimators* compared against
   the analytical results.
- `franke.py` -- the complete study of Section *A complete example: the Franke function*:
   design matrices for two-dimensional polynomials, the train-test
   split, cross-validated selection of $\lambda$, test error against
   polynomial degree for all three methods, and the count of non-zero
   Lasso coefficients.
- `scaling.py` -- the intercept and standardisation conventions
   of Section *Scaling, centring and the intercept*, with a reconciliation of a
   hand-written implementation against `scikit-learn`.

Each file runs as a script and reproduces the numbers quoted in this chapter.
An executable version is available as a Jupyter notebook in the accompanying
Jupyter-book.


## Exercises

### Warm-up exercises

1. **The normal equations by hand.**
   For the toy problem of Eq. (3.46):
   (a) compute $\bm{X}^{T}\bm{X}$ and $\bm{X}^{T}\bm{y}$ and verify
   Eq. (3.48);
   (b) compute the hat matrix (3.9) and verify that it is
   symmetric, idempotent, and has trace $2$;
   (c) verify that the residual is orthogonal to both columns of $\bm{X}$.
2. **Ridge by hand.**
   For the same problem, derive Eq. (3.49) by differentiating
   the penalised cost, and explain why the two coefficients are shrunk by
   different factors.  Relate the factors to the singular values of $\bm{X}$.
3. **The Lasso threshold.**
   For the same problem, show that the Lasso sets $\theta_1=0$ for
   $\lambda\ge4$ and $\theta_0=0$ for $\lambda\ge16$ (with the convention of
   Eq. (3.36) and the $1/n$ dropped).  Sketch all three
   coefficient paths on one plot.
4. **Equivariance under rescaling.**
   (a) Show that if a column of $\bm{X}$ is multiplied by $c$, the OLS fitted
   values $\tilde{\bm{y}}$ are unchanged while the corresponding
   coefficient is divided by $c$.
   (b) Show that this fails for Ridge, and explain in one sentence why
   standardisation is therefore mandatory for penalised regression and
   optional for OLS.
5. **Maximum likelihood with a different noise model.**
   Repeat the derivation of Section *Deriving least squares from a probability distribution* assuming Laplace
   noise, $p(\varepsilon)\propto\exp(-|\varepsilon|/b)$.  Which cost function
   results?  What does this say about the robustness of the two fits to
   outliers?
6. **Degrees of freedom.**
   (a) Show that $\mathrm{Tr}(\bm{H})=p$ for the OLS hat matrix.
   (b) Show that the Ridge analogue
   $\bm{H}_\lambda=\bm{X}(\bm{X}^{T}\bm{X}+\lambda\bm{I})^{-1}\bm{X}^{T}$
   has trace $\sum_i\sigma_i^{2}/(\sigma_i^{2}+\lambda)$, which is
   Eq. (1.132).
   (c) Verify that $\bm{H}_\lambda$ is symmetric but *not* idempotent for
   $\lambda>0$, and interpret this using the eigenvalue argument of
   Section *Orthonormal bases and projections*.
7. **Conditioning in practice (numerical).**
   Fit a polynomial of degree $d$ to $100$ points on $[0,1]$ using both
   functions of Section *Ordinary least squares*.
   (a) For $d=2,4,\dots,14$, compare the coefficients returned by the two.
   (b) Plot $\kappa_2(\bm{X})$ and $\kappa_2(\bm{X}^{T}\bm{X})$ against $d$.
   (c) At which $d$ do the two methods first disagree in the first significant
   digit, and how does that relate to Eq. (1.118)?
8. **Coordinate descent (numerical).**
   Implement the Lasso by coordinate descent as in
   Eq. (3.45).
   (a) Verify on an orthonormal design that it reproduces the soft
   thresholding result (3.42) exactly.
   (b) Compare against `sklearn.linear_model.Lasso` on a general
   design, taking care over the factor conventions noted in
   Section *Comparing the three estimators*.
   (c) Plot the number of non-zero coefficients against $\lambda$.
9. **Confidence intervals (numerical).**
   For data generated from a known linear model:
   (a) compute the confidence intervals (3.24) for each
   coefficient;
   (b) repeat the whole experiment $1000$ times with fresh noise and count how
   often the true value lies inside the interval;
   (c) does the coverage match the nominal $95\%$?  Repeat with two nearly
   collinear columns and comment.
10. **Ridge and the bias-variance trade-off (numerical).**
   For a fixed polynomial degree, plot the squared bias, the variance and the
   test error of a Ridge fit against $\lambda$, using the bootstrap procedure
   of Section *The bias-variance tradeoff*.  Verify that the bias increases
   monotonically, the variance decreases monotonically, and the minimum of the
   sum occurs at $\lambda>0$.

### Project-style exercise: regression analysis of the Franke function

This extended exercise assembles the whole chapter, and follows the structure
of the first project of the course.

**Part a: ordinary least squares.** 
Generate a data set from the Franke function (3.56) with
$x,y\in[0,1]$ and added noise $\mathcal{N}(0,\sigma^{2})$.  Write your own
code -- using either a matrix inversion or, preferably, the singular value
decomposition -- and perform a standard least-squares analysis with
polynomials in $x$ and $y$ up to fifth order.

**Part b: the bias-variance trade-off with the bootstrap.** 
Using the bootstrap of Section *Resampling: the jackknife and the bootstrap*, decompose the test error
into bias and variance as in Eq. (2.47) and plot all three
against polynomial degree.  Explain the shape of each curve.  Then repeat with
a substantially larger data set and account for the shift in the optimum.

**Part c: cross-validation.** 
Implement $k$-fold cross-validation from scratch, with $k$ between five and
ten, and compare its estimate of the test error with the bootstrap estimate of
part b.  Which is cheaper, and which has the smaller variance?

**Part d: Ridge regression.** 
Repeat parts a to c for Ridge regression, as a function of both the polynomial
degree and the penalty $\lambda$.  Present your results as a heat map of the
cross-validated test error over the two hyperparameters, and identify the
optimum.  Discuss the results in the light of the
shrinkage (3.31) and of the variance
difference (3.35).

**Part e: the Lasso.** 
Repeat part d for the Lasso, using either your own coordinate-descent
implementation or `scikit-learn`.  In addition, count the non-zero
coefficients at the optimal $\lambda$ and compare with the number retained by
Ridge.  Which monomials survive, and does the selection agree with your
expectation from the shape of the Franke surface?

**Part f: real data.** 
Finally, replace the Franke function by a real data set -- digital terrain
data are a natural choice, being a genuine two-dimensional surface -- and
repeat the analysis.  Discuss what changes when the true function is unknown,
the noise level is unknown, and the observations may be spatially correlated.
In particular, reconsider whether a random train-test split is defensible, in
the light of Section *Correlated data and the autocorrelation function*.
